# DysonianLineCNN — Train and Evaluate

Thin wrapper around [`dyson_cnn`](../dyson_cnn/). All real logic lives in the Python package;
this notebook exists only to pin the training invocation and surface results in a Jupyter UI.

On a fresh Colab runtime the first cell clones this **public** repository
automatically — no keys or secrets needed. On Mac this is a no-op — just run the cells.

Profile is selected via `config/training.json` `active_profile` field, or the `DYSON_PROFILE`
environment variable (which takes precedence). Defaults to `colab_full` on Colab, set
`DYSON_PROFILE=mac_dev` on Mac for fast debug iterations.

## 1. Environment setup

Idempotent across Google Colab and local Mac.

**On Colab** (`google.colab` module detected):
1. Clone (or pull) the public repo `https://github.com/AndriiUriadov/DysonianLineCNN.git`
   into `/content/DysonianLineCNN` — no SSH key or Colab secret required.
2. `pip install -e . --no-deps` the package in editable mode so imports work
   from any cwd while preserving Colab's curated TensorFlow stack.
3. Mount Google Drive at `/content/drive`.

**On local Mac**: skip the Colab bootstrap; just verify that the notebook is
running from the repository root so `import dyson_cnn` resolves.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # --- Colab-only: public git clone -> pip install -> drive mount ---
    # The repository is public, so no SSH key or Colab secret is needed.
    repo_dir = '/content/DysonianLineCNN'
    if not os.path.isdir(repo_dir):
        get_ipython().system('git clone https://github.com/AndriiUriadov/DysonianLineCNN.git ' + repo_dir)
    else:
        get_ipython().system('cd ' + repo_dir + ' && git pull')

    os.chdir(repo_dir)
    get_ipython().system('pip install -q -e . --no-deps')

    from google.colab import drive
    drive.mount('/content/drive')

# Verify we are at the repo root (works on both Colab and local Mac)
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'dyson_cnn').is_dir():
    for parent in Path.cwd().parents:
        if (parent / 'dyson_cnn').is_dir():
            REPO_ROOT = parent
            os.chdir(REPO_ROOT)
            break
    else:
        raise RuntimeError(f'Could not find repo root from {Path.cwd()}')

print(f'Environment : {"Colab" if IN_COLAB else "local Mac"}')
print(f'Repo root   : {REPO_ROOT}')

## 2. Load all configs

Resolves `paths.json`, `dataset.json` (or `sets/<set_name>.json`), `training.json`,
`inference.json`. Profile is taken from `DYSON_PROFILE` if set, else
`training.json.active_profile`.

Set `SET_NAME` to a set identifier (e.g. `"set-1"`) to use per-set dataset config and
Drive subdirectory. Leave `None` for legacy single-dataset mode.

In [ ]:
from dyson_cnn.config import load_all

SET_NAME = "set-1"  # e.g. "set-1", "set-2", ... or None for legacy mode

paths, dataset_cfg, training_cfg, inference_cfg = load_all(
    REPO_ROOT / 'config',
    profile=os.environ.get('DYSON_PROFILE'),
    set_name=SET_NAME,
)

print(f'Runtime         : {paths["runtime"]}')
print(f'Set name        : {SET_NAME or "(legacy)"}')
print(f'Project dir     : {paths.get("set_project_dir", paths["project_dir"])}')
print(f'Profile         : {training_cfg["profile_name"]}')
print(f'Epochs / batch  : {training_cfg["epochs"]} / {training_cfg["batch_size"]}')
print(f'max_samples     : {training_cfg["max_samples"]}')
print(f'mixed_precision : {training_cfg["mixed_precision"]}')

## 3. Train

`train_and_save_run` is the single entry point. It loads the dataset, splits it, builds the
CNN, fits, and writes an atomic run directory containing:

- `cnn_model.keras`
- `y_min.npy` / `y_max.npy` (from train split only)
- `B_axis.npy` / `B_axis.csv`
- `model_meta.json` (generator metadata snapshot + training profile)
- `history.csv`
- `loss.png` / `loss_full_and_zoom.png`
- `_X_test.npy` / `_y_test.npy` (cached for step 4)

In [ ]:
from dyson_cnn.train import train_and_save_run

run_dir = train_and_save_run(paths, dataset_cfg, training_cfg)
print(f'\nSaved run: {run_dir}')

## 4. Evaluate on the test split

`evaluate_run` loads the saved model and the cached test split from the run directory and
produces per-head MAE/RMSE plus a parity plot.

In [ ]:
from dyson_cnn.evaluate import evaluate_run

result = evaluate_run(run_dir)
print(f'\nN test samples: {result["n_samples"]}')
for head, m in result['metrics'].items():
    print(f'  {head:>2}: MAE={m["mae"]:.6g}, RMSE={m["rmse"]:.6g}')

## 5. Next steps

1. Open [`config/inference.json`](../config/inference.json) and set `runName` to the newly
   created run's timestamp directory name.
2. Run [`02_infer_real.ipynb`](02_infer_real.ipynb) to predict Dysonian parameters from a
   real experimental spectrum CSV.
3. Open `matlab/Validator.m` to reconstruct the fitted curve and overlay it on the raw spectrum.